In [1]:
'''
ParallelAgent: You'll unleash efficiency by running multiple agents simultaneously and then synthesizing their collective findings into a single, comprehensive answer.

ParallelAgent. This workflow agent executes a list of sub-agents concurrently, dramatically speeding up tasks that can be performed independently.

Our New Workflow: The Multi-Researcher

Parallel Agent: Simultaneously runs three specialist agents:
MuseumFinderAgent: Finds a museum.
ConcertFinderAgent: Finds a concert.
FoodieAgent: Finds a restaurant.
Synthesis Agent: Once all three parallel searches are complete, this final agent gathers the results (which were saved to the shared state) and formats them into a single, neat summary for the user.
This pattern lets us get a lot of work done, fast
'''


"\nParallelAgent: You'll unleash efficiency by running multiple agents simultaneously and then synthesizing their collective findings into a single, comprehensive answer.\n\nParallelAgent. This workflow agent executes a list of sub-agents concurrently, dramatically speeding up tasks that can be performed independently.\n\nOur New Workflow: The Multi-Researcher\n\nParallel Agent: Simultaneously runs three specialist agents:\nMuseumFinderAgent: Finds a museum.\nConcertFinderAgent: Finds a concert.\nFoodieAgent: Finds a restaurant.\nSynthesis Agent: Once all three parallel searches are complete, this final agent gathers the results (which were saved to the shared state) and formats them into a single, neat summary for the user.\nThis pattern lets us get a lot of work done, fast\n"

In [2]:
import os
import sys
import  json
import asyncio
import random
import string
from uuid import uuid4
from typing import List,Any
from IPython.display import HTML, Markdown, display

#----------ADK , Agent and Evaluation components Tools Contextimports here------------------

from google.adk.agents import Agent, SequentialAgent,ParallelAgent,LoopAgent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search, ToolContext
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content , Part

from dotenv import load_dotenv

print(" All libraries are imported!")

 All libraries are imported!


In [3]:
load_dotenv()

True

In [4]:
#Runner to Help run the agent: This is a HELPER function
async def run_agent_query(agent:Agent,query:str,session : Session,user_id: str,is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n Running query for agent: '{agent.name}' in session: '{session.id}'...")
    runner = Runner(
        agent = agent,
        session_service = session_service,
        app_name = agent.name)
    final_response = ""
    try:
        async for event in runner.run_async(user_id = user_id ,session_id = session.id,new_message = Content(parts=[Part(text = query)],role ="user")):
            if not is_router:
                # Let's see what the agent is thinking! through events 
                print(f"EVENT:{event}")
                if event.is_final_response():
                    final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
        print("\n" + "-"*50)
        print("✅ Final Response:")
        display(Markdown(final_response))
        print("-"*50 + "\n")
    return final_response

In [5]:
# --- Initializing Session Service ---
session_service = InMemorySessionService()
my_user_id = "adk_user_001"

In [6]:
db_agent = Agent(
    name = "db_agent",
    model = "gemini-3.5-flash",
    instruction = "You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}"
)

In [ ]:
#-------------------------Parallel Workflow--------------------------------------------------------------------------------------------
# Specialist Agent 1
museum_finder_agent = Agent(
    name="museum_finder_agent", 
    model="gemini-3.5-flash", 
    tools=[google_search],
    instruction="You are a museum expert. Find the best museum based on the user's query. Output only the museum's name.",
    output_key="museum_result"
)

# Specialist Agent 2
concert_finder_agent = Agent(
    name="concert_finder_agent", 
    model="gemini-3.5-flash", 
    tools=[google_search],
    instruction="You are an events guide. Find a concert based on the user's query. Output only the concert name and artist.",
    output_key="concert_result"
)

restaurant_finder_agent = Agent(
    name="restaurant_finder_agent",
    model="gemini-3.5-flash",
    tools=[google_search],
    instruction="""
    You are an expert food critic. Your goal is to find the best restaurant based on a user's request.
    When you recommend a place, you must output *only* the name of the establishment.
    For example, if the best sushi is at 'Jin Sho', you should output only: Jin Sho
    """,
    output_key="restaurant_result" 
)

# ----------------------The ParallelAgent runs all three specialists at once-------------------------------
parallel_research_agent = ParallelAgent(
    name="parallel_research_agent",
    sub_agents=[museum_finder_agent, concert_finder_agent, restaurant_finder_agent]
)

# Agent to synthesize the parallel results
synthesis_agent = Agent(
    name="synthesis_agent", 
    model="gemini-3.5-flash",
    instruction="""
    You are a helpful assistant. Combine the following research results into a clear, bulleted list for the user.
    - Museum: {museum_result}
    - Concert: {concert_result}
    - Restaurant: {restaurant_result}
    """
)

# ---------- The SequentialAgent runs the parallel search, then the synthesis --------------------

parallel_planner_agent = SequentialAgent(
    name="parallel_planner_agent",
    sub_agents=[parallel_research_agent, synthesis_agent],
    description="A workflow that finds multiple things in parallel and then summarizes the results."
)

print(" Agent team supercharged with a ParallelAgent workflow!")